In [1]:
# =========================================================
# STEP 3 — EXPLORATORY DATA ANALYSIS (EDA)
# + COMPLAINT INTELLIGENCE ANALYTICS
# FILE: notebooks/02_eda_complaint_analytics.ipynb
# PROJECT: Smart Complaint Analytics System
# INPUT : data/processed/clean_complaints.csv
# =========================================================

# =========================================================
# 3.1 IMPORT LIBRARIES
# =========================================================

import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

warnings.filterwarnings("ignore")

# Optional: better font rendering
plt.rcParams["font.family"] = "DejaVu Sans"

In [2]:
# =========================================================
# 3.2 LOAD CLEAN DATASET
# =========================================================

DATA_PATH = Path("../data/processed/clean_complaints.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["created_at"])

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 25)


,complaint_id,created_at,channel,citizen_comment,complaint_category,department,district,village,sentiment,urgency_level,...,engagement_score,year,month,month_name,quarter,day_of_week,is_resolved,sla_met_flag,is_critical,priority_bucket
0,CMP-04708,2024-01-01 03:40:36,Call Center,Tempat pembuangan di Dadaprejo sudah penuh.,Waste Management,Dinas Lingkungan Hidup,Junrejo,Dadaprejo,Negative,Medium,...,3877,2024,1,January,1,Monday,1,0,0,Medium
1,CMP-07774,2024-01-01 03:43:25,Call Center,Aspal rusak parah di daerah Temas dan belum di...,Road Infrastructure,Dinas PUPR,Batu,Temas,Negative,Low,...,845,2024,1,January,1,Monday,1,0,0,Low
2,CMP-01189,2024-01-01 06:35:30,LAPOR!,Air PDAM di Mojorejo keruh dan tidak layak dig...,Clean Water,PDAM,Bumiaji,Mojorejo,Neutral,Medium,...,1187,2024,1,January,1,Monday,0,0,0,Medium
3,CMP-01853,2024-01-01 10:09:18,Website,Genangan air sering terjadi di Dadaprejo saat ...,Flooding,Dinas PUPR,Junrejo,Dadaprejo,Neutral,High,...,4440,2024,1,January,1,Monday,0,0,0,High
4,CMP-07824,2024-01-01 14:43:25,TikTok,Lampu jalan rusak di kawasan Gunungsari.,Street Lighting,Dinas Perhubungan,Junrejo,Gunungsari,Negative,Medium,...,1780,2024,1,January,1,Monday,1,1,0,Medium


In [3]:
# =========================================================
# 3.3 EXECUTIVE KPI CALCULATION
# =========================================================

total_complaints = len(df)
resolution_rate = df["is_resolved"].mean() * 100
sla_compliance = df["sla_met_flag"].mean() * 100
critical_rate = df["is_critical"].mean() * 100
avg_response_time = df["response_time_hours"].mean()

top_category = (
    df["complaint_category"]
    .value_counts()
    .idxmax()
)

top_department = (
    df["department"]
    .value_counts()
    .idxmax()
)

print("=" * 60)
print("EXECUTIVE KPI SUMMARY")
print("=" * 60)
print(f"Total Complaints   : {total_complaints:,}")
print(f"Resolution Rate    : {resolution_rate:.2f}%")
print(f"SLA Compliance     : {sla_compliance:.2f}%")
print(f"Critical Rate      : {critical_rate:.2f}%")
print(f"Avg Response Time  : {avg_response_time:.2f} hours")
print(f"Top Category       : {top_category}")
print(f"Top Department     : {top_department}")

EXECUTIVE KPI SUMMARY
Total Complaints   : 10,000
Resolution Rate    : 47.79%
SLA Compliance     : 38.68%
Critical Rate      : 8.01%
Avg Response Time  : 35.73 hours
Top Category       : Road Infrastructure
Top Department     : Dinas PUPR


In [4]:
# =========================================================
# 3.4 MONTHLY COMPLAINT TREND
# =========================================================

monthly_trend = (
    df.groupby(["year", "month"])
      .size()
      .reset_index(name="total_complaints")
)

monthly_trend["period"] = (
    monthly_trend["year"].astype(str)
    + "-"
    + monthly_trend["month"].astype(str).str.zfill(2)
)

fig = px.line(
    monthly_trend,
    x="period",
    y="total_complaints",
    markers=True,
    title="Monthly Complaint Trend",
    template="plotly_white"
)

fig.update_layout(
    title_font_size=22,
    xaxis_title="Period",
    yaxis_title="Number of Complaints"
)

fig.show()

# Save screenshot as:
# assets/monthly-complaint-trend.png

In [5]:
# =========================================================
# 3.5 COMPLAINT CATEGORY DISTRIBUTION
# =========================================================

category_df = (
    df["complaint_category"]
    .value_counts()
    .reset_index()
)

category_df.columns = ["Category", "Count"]

fig = px.bar(
    category_df,
    x="Count",
    y="Category",
    orientation="h",
    title="Complaint Category Distribution",
    template="plotly_white",
    color="Count",
    color_continuous_scale="Blues"
)

fig.update_layout(
    title_font_size=22,
    yaxis=dict(categoryorder="total ascending")
)

fig.show()

# Save screenshot as:
# assets/complaint-category-distribution.png

In [6]:
# =========================================================
# 3.6 URGENCY LEVEL DISTRIBUTION
# =========================================================

urgency_df = (
    df["urgency_level"]
    .value_counts()
    .reindex(["Low", "Medium", "High", "Critical"], fill_value=0)
    .reset_index()
)

urgency_df.columns = ["Urgency", "Count"]

fig = px.pie(
    urgency_df,
    names="Urgency",
    values="Count",
    hole=0.65,
    title="Urgency Level Distribution",
    template="plotly_white"
)

fig.show()

# Save screenshot as:
# assets/urgency-distribution.png

In [7]:
# =========================================================
# 3.7 SLA COMPLIANCE BY DEPARTMENT
# =========================================================

sla_department = (
    df.groupby("department")["sla_met_flag"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .reset_index()
)

sla_department.columns = ["Department", "SLA Compliance (%)"]

fig = px.bar(
    sla_department,
    x="Department",
    y="SLA Compliance (%)",
    title="SLA Compliance by Department",
    template="plotly_white",
    color="SLA Compliance (%)",
    color_continuous_scale="Viridis"
)

fig.update_layout(title_font_size=22)

fig.show()

# Save screenshot as:
# assets/sla-compliance-by-department.png

In [8]:
# =========================================================
# 3.8 RESOLUTION STATUS DISTRIBUTION
# =========================================================

status_df = (
    df["status"]
    .value_counts()
    .reset_index()
)

status_df.columns = ["Status", "Count"]

fig = px.bar(
    status_df,
    x="Status",
    y="Count",
    title="Complaint Resolution Status",
    template="plotly_white",
    color="Status"
)

fig.update_layout(title_font_size=22)

fig.show()

# Save screenshot as:
# assets/complaint-resolution-status.png

In [16]:
# =========================================================
# 3.9 GEOGRAPHIC COMPLAINT INTELLIGENCE
# =========================================================

district_df = (
    df["district"]
    .value_counts()
    .reset_index()
)

district_df.columns = ["District", "Count"]

fig = px.bar(
    district_df,
    x="District",
    y="Count",
    title="Complaint Distribution by District",
    template="plotly_white",
    color="Count",
    color_continuous_scale="Plasma"
)

fig.update_layout(title_font_size=22)

fig.show()

# Save screenshot as:
# assets/complaints-by-district.png

In [17]:
# =========================================================
# 3.10 TOP 10 HIGH PRIORITY COMPLAINTS
# =========================================================

top_priority = (
    df.sort_values("priority_score", ascending=False)
      [["created_at", "complaint_category", "department",
        "urgency_level", "priority_score", "status"]]
      .head(10)
)

top_priority

,created_at,complaint_category,department,urgency_level,priority_score,status
4046,2024-10-16 14:29:01,Administrative Services,Dinas Kependudukan,Critical,100,In Progress
303,2024-01-22 15:17:52,Flooding,Dinas PUPR,Critical,100,In Progress
9567,2025-11-26 00:57:59,Flooding,Dinas PUPR,Critical,100,Open
4858,2024-12-15 15:43:00,Administrative Services,Dinas Kependudukan,Critical,100,In Progress
5586,2025-02-04 07:48:13,Tourism Services,Dinas Pariwisata,Critical,100,Open
1089,2024-03-20 16:28:31,Flooding,Dinas PUPR,Critical,100,Resolved
6823,2025-05-07 10:23:29,Waste Management,Dinas Lingkungan Hidup,Critical,100,Open
5588,2025-02-04 12:40:22,Waste Management,Dinas Lingkungan Hidup,Critical,100,Resolved
2688,2024-07-07 20:25:02,Public Safety,Satpol PP,Critical,100,In Progress
2697,2024-07-08 16:42:25,Public Safety,Satpol PP,Critical,100,Resolved


In [18]:
# =========================================================
# 3.11 EXECUTIVE INSIGHT SUMMARY
# =========================================================

print(f"""
============================================================
SMART COMPLAINT ANALYTICS SYSTEM
EDA EXECUTIVE SUMMARY
============================================================

Total Complaints        : {total_complaints:,}
Resolution Rate         : {resolution_rate:.2f}%
SLA Compliance          : {sla_compliance:.2f}%
Critical Complaint Rate : {critical_rate:.2f}%
Average Response Time   : {avg_response_time:.2f} hours

Top Complaint Category  : {top_category}
Top Department          : {top_department}

Strategic Insight:
The analysis reveals the operational areas receiving the
highest complaint volumes and identifies departments with
the strongest and weakest SLA performance. These insights
can be used to prioritize service improvements, allocate
resources, and strengthen citizen satisfaction.
============================================================
""")


SMART COMPLAINT ANALYTICS SYSTEM
EDA EXECUTIVE SUMMARY

Total Complaints        : 10,000
Resolution Rate         : 47.79%
SLA Compliance          : 38.68%
Critical Complaint Rate : 8.01%
Average Response Time   : 35.73 hours

Top Complaint Category  : Road Infrastructure
Top Department          : Dinas PUPR

Strategic Insight:
The analysis reveals the operational areas receiving the
highest complaint volumes and identifies departments with
the strongest and weakest SLA performance. These insights
can be used to prioritize service improvements, allocate
resources, and strengthen citizen satisfaction.



In [9]:
# =========================================================
# ELITE DARK THEME FOR STEP 3 VISUALIZATIONS
# Jalankan SEKALI di awal notebook STEP 3
# =========================================================

import plotly.io as pio

# Premium Dark Theme Configuration
DARK_BG = "#061226"       # Deep navy background
PAPER_BG = "#081a33"      # Slightly lighter navy
CARD_BG = "#0d1b3d"
GRID_COLOR = "rgba(255,255,255,0.10)"
FONT_COLOR = "#E8F1FF"
ACCENT_BLUE = "#60A5FA"

# Set Plotly default template
pio.templates.default = "plotly_dark"

# Common layout settings
COMMON_LAYOUT = dict(
    paper_bgcolor=PAPER_BG,
    plot_bgcolor=DARK_BG,
    font=dict(
        family="Inter, Segoe UI, sans-serif",
        color=FONT_COLOR,
        size=14
    ),
    title=dict(
        font=dict(size=24, color="#FFFFFF"),
        x=0.02,
        xanchor="left"
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor=GRID_COLOR,
        zeroline=False,
        showline=False
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=GRID_COLOR,
        zeroline=False,
        showline=False
    ),
    legend=dict(
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(color=FONT_COLOR)
    ),
    margin=dict(l=60, r=40, t=80, b=60)
)

print("Elite Dark Theme activated successfully.")

Elite Dark Theme activated successfully.


In [10]:
fig = px.line(
    monthly_trend,
    x="period",
    y="total_complaints",
    markers=True,
    title="📈 Monthly Complaint Trend",
    template="plotly_dark"
)

fig.update_traces(
    line=dict(color="#3B82F6", width=4),
    marker=dict(size=8, color="#60A5FA")
)

fig.update_layout(
    **COMMON_LAYOUT,
    xaxis_title="Period",
    yaxis_title="Number of Complaints"
)

fig.show()

In [12]:
fig = px.bar(
    category_df,
    x="Count",
    y="Category",
    orientation="h",
    title="📊 Complaint Category Distribution",
    template="plotly_dark",
    color="Count",
    color_continuous_scale="Plasma"
)

fig = px.bar(
    category_df,
    x="Count",
    y="Category",
    orientation="h",
    title="📊 Complaint Category Distribution",
    template="plotly_dark",
    color="Count",
    color_continuous_scale="Plasma"
)

fig.update_layout(
    **COMMON_LAYOUT,
    coloraxis_colorbar=dict(
        bgcolor="rgba(0,0,0,0)",
        tickfont=dict(color=FONT_COLOR)
    )
)

fig.update_yaxes(
    categoryorder="total ascending",
    showgrid=False
)

fig.show()

fig.show()

In [13]:
fig = px.pie(
    urgency_df,
    names="Urgency",
    values="Count",
    hole=0.65,
    title="🚨 Urgency Level Distribution",
    template="plotly_dark",
    color="Urgency",
    color_discrete_map={
        "Low": "#22C55E",
        "Medium": "#EAB308",
        "High": "#F97316",
        "Critical": "#EF4444"
    }
)

fig.update_layout(**COMMON_LAYOUT)

fig.update_traces(
    textinfo="percent+label",
    textfont=dict(color="white", size=14),
    marker=dict(line=dict(color=PAPER_BG, width=2))
)

fig.show()

In [14]:
fig = px.bar(
    sla_department,
    x="Department",
    y="SLA Compliance (%)",
    title="🏛️ SLA Compliance by Department",
    template="plotly_dark",
    color="SLA Compliance (%)",
    color_continuous_scale="Viridis"
)

fig.update_layout(
    **COMMON_LAYOUT,
    xaxis_tickangle=-20,
    coloraxis_colorbar=dict(
        bgcolor="rgba(0,0,0,0)",
        tickfont=dict(color=FONT_COLOR)
    )
)

fig.show()

In [19]:
# =========================================================
# 3.8 RESOLUTION STATUS DISTRIBUTION (ELITE DARK THEME)
# =========================================================

fig = px.bar(
    status_df,
    x="Status",
    y="Count",
    title="📋 Complaint Resolution Status",
    template="plotly_dark",
    color="Status",
    color_discrete_map={
        "Resolved": "#22C55E",      # Green
        "In Progress": "#EAB308",   # Yellow
        "Open": "#EF4444"           # Red
    }
)

fig.update_traces(
    texttemplate="%{y:,}",
    textposition="outside"
)

fig.update_layout(
    **COMMON_LAYOUT,
    xaxis_title="Resolution Status",
    yaxis_title="Number of Complaints",
    showlegend=False
)

fig.show()

# Save screenshot as:
# assets/complaint-resolution-status.png

In [20]:
# =========================================================
# 3.9 GEOGRAPHIC COMPLAINT INTELLIGENCE (ELITE DARK THEME)
# =========================================================

fig = px.bar(
    district_df,
    x="District",
    y="Count",
    title="🗺️ Complaint Distribution by District",
    template="plotly_dark",
    color="Count",
    color_continuous_scale="Turbo"
)

fig.update_traces(
    texttemplate="%{y:,}",
    textposition="outside"
)

fig.update_layout(
    **COMMON_LAYOUT,
    xaxis_title="District",
    yaxis_title="Number of Complaints",
    coloraxis_colorbar=dict(
        bgcolor="rgba(0,0,0,0)",
        tickfont=dict(color=FONT_COLOR),
        title=dict(
            text="Complaints",
            font=dict(color=FONT_COLOR)
        )
    )
)

fig.show()

# Save screenshot as:
# assets/complaints-by-district.png

In [15]:
top_priority = (
    df.sort_values("priority_score", ascending=False)
      [[
          "created_at",
          "complaint_category",
          "department",
          "urgency_level",
          "priority_score",
          "status"
      ]]
      .head(10)
)

# Styling for Jupyter Notebook
top_priority.style \
    .background_gradient(
        subset=["priority_score"],
        cmap="Reds"
    ) \
    .set_properties(**{
        "background-color": "#081a33",
        "color": "white",
        "border": "1px solid #1f3b73"
    }) \
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#0f172a"),
                ("color", "#60A5FA"),
                ("font-weight", "bold"),
                ("text-align", "center")
            ]
        }
    ])

,created_at,complaint_category,department,urgency_level,priority_score,status
4046,2024-10-16 14:29:01,Administrative Services,Dinas Kependudukan,Critical,100,In Progress
303,2024-01-22 15:17:52,Flooding,Dinas PUPR,Critical,100,In Progress
9567,2025-11-26 00:57:59,Flooding,Dinas PUPR,Critical,100,Open
4858,2024-12-15 15:43:00,Administrative Services,Dinas Kependudukan,Critical,100,In Progress
5586,2025-02-04 07:48:13,Tourism Services,Dinas Pariwisata,Critical,100,Open
1089,2024-03-20 16:28:31,Flooding,Dinas PUPR,Critical,100,Resolved
6823,2025-05-07 10:23:29,Waste Management,Dinas Lingkungan Hidup,Critical,100,Open
5588,2025-02-04 12:40:22,Waste Management,Dinas Lingkungan Hidup,Critical,100,Resolved
2688,2024-07-07 20:25:02,Public Safety,Satpol PP,Critical,100,In Progress
2697,2024-07-08 16:42:25,Public Safety,Satpol PP,Critical,100,Resolved
